In [ ]:
# %pip install MetaTrader5 pandas requests

import MetaTrader5 as mt5
import pandas as pd
import requests
import time
import traceback
from datetime import datetime

# ==========================================
# ⚙️ USER CONFIGURATION
# ==========================================

# 1. ACCOUNT CREDENTIALS (FOR AUTO-LOGIN)
# ------------------------------------------
# Leave these 0/None if you are already logged into the MT5 Terminal.
LOGIN_ID =2707383   # Example: 12345678
PASSWORD = "Ducanh@6"
SERVER = "Headway-Demo" # Example: "ICMarkets-Demo"

# 2. SECURITY & ALERTS
# ------------------------------------------
WEBHOOK_URL = "https://discord.com/api/webhooks/1463101858791952554/az4l16sYfAZ9mrlS7cQ_F8mDvkRM68Cfd4kXDss5W9LzRvoti7RdvadL32cXDIOrKOlI"
DEMO_ONLY = True         # 🔒 SAFETY LOCK: Stops bot if account is Real Money

# 3. TRADING SETTINGS (BITCOIN SPECIFIC)
# ------------------------------------------
SYMBOL = "BTCUSD"        # Check Market Watch (e.g., "BTCUSD", "Bitcoin")
TIMEFRAME = mt5.TIMEFRAME_M15
LOT_SIZE = 0.01          # Start small for BTC (0.01 lots)
MAGIC_NUMBER = 888888    # Unique ID for this bot
DEVIATION = 50           # Higher slippage allowed for Crypto volatility

# 4. SCHEDULE (24-Hour Clock)
# ------------------------------------------
# Crypto runs 24/7, but we can stick to "Waking Hours" to monitor it.
START_HOUR = 7   # 7:00 AM
END_HOUR = 23    # 11:00 PM

# ==========================================
# 📡 MODULE 1: DISCORD NOTIFIER
# ==========================================
def send_alert(message, type="info"):
    if "YOUR_NEW" in WEBHOOK_URL: return
    colors = {"info": 3447003, "trade": 5763719, "error": 15548997}
    
    payload = {
        "username": "BTC Demo Bot ₿",
        "embeds": [{
            "description": message,
            "color": colors.get(type, 3447003),
            "timestamp": datetime.utcnow().isoformat()
        }]
    }
    try:
        requests.post(WEBHOOK_URL, json=payload)
    except Exception:
        pass

# ==========================================
# 🔐 MODULE 2: CONNECTION & SAFETY
# ==========================================
def connect_mt5():
    # 1. Initialize
    if not mt5.initialize():
        print(f"❌ MT5 Failed: {mt5.last_error()}")
        return False
    
    # 2. Login (If credentials provided)
    if LOGIN_ID != 0:
        authorized = mt5.login(LOGIN_ID, password=PASSWORD, server=SERVER)
        if not authorized:
            print(f"❌ Login Failed: {mt5.last_error()}")
            return False

    # 3. 🔒 DEMO ACCOUNT CHECK (SAFETY LOCK)
    account_info = mt5.account_info()
    if account_info is None:
        print("❌ Could not get account info")
        return False

    print(f"✅ Logged in as: {account_info.name} ({account_info.login})")
    
    if DEMO_ONLY:
        # Check leverage or trade mode. 
        # Note: 'trade_mode' 0=Demo, 2=Real (varies by broker API version).
        # Safer check: Look for "Demo" in the server name or user flag.
        if account_info.trade_mode == mt5.ACCOUNT_TRADE_MODE_REAL:
            msg = "⛔ SAFETY STOP: Real Money Account Detected! Bot is in DEMO_ONLY mode."
            print(msg)
            send_alert(msg, "error")
            mt5.shutdown()
            return False
        else:
            print("✅ Safety Check Passed: Demo Account Detected.")
            
    return True

# ==========================================
# 💹 MODULE 3: MARKET DATA & EXECUTION
# ==========================================
def get_market_data(symbol, n=100):
    rates = mt5.copy_rates_from_pos(symbol, TIMEFRAME, 0, n)
    if rates is None: return None
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    return df

def execute_trade(action, symbol):
    tick = mt5.symbol_info_tick(symbol)
    if not tick: return
    
    # BTC Spread can be wide, ensure we have a valid price
    price = tick.ask if action == 'buy' else tick.bid
    type_trade = mt5.ORDER_TYPE_BUY if action == 'buy' else mt5.ORDER_TYPE_SELL
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": LOT_SIZE,
        "type": type_trade,
        "price": price,
        "deviation": DEVIATION,
        "magic": MAGIC_NUMBER,
        "comment": "BTC Demo Bot",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    res = mt5.order_send(request)
    
    if res.retcode != mt5.TRADE_RETCODE_DONE:
        msg = f"❌ Trade Failed: {res.comment}"
        print(msg)
        send_alert(msg, "error")
    else:
        msg = f"🚀 **BTC {action.upper()} EXECUTED** @ {price}\nSize: {LOT_SIZE}"
        print(msg)
        send_alert(msg, "trade")

# ==========================================
# 🧠 MODULE 4: MAIN LOOP
# ==========================================
def run_bot():
    if not connect_mt5(): return

    print(f"🤖 BTC Bot Online. Monitoring {SYMBOL}...")
    send_alert(f"₿ Bitcoin Bot Started on Demo Account.", "info")
    
    is_sleeping = False
    
    try:
        while True:
            # --- 1. SCHEDULE CHECK ---
            now = datetime.now()
            # BTC trades on weekends, so we removed the weekend check!
            # We only check hours now.
            is_active_hours = START_HOUR <= now.hour < END_HOUR
            
            if not is_active_hours:
                if not is_sleeping:
                    send_alert(f"💤 Pausing for the night. Back at {START_HOUR}:00.", "info")
                    is_sleeping = True
                time.sleep(60)
                continue
            
            if is_sleeping:
                send_alert("☀️ Waking up. Resuming BTC scans.", "info")
                is_sleeping = False

            # --- 2. LOGIC (SMA CROSSOVER) ---
            df = get_market_data(SYMBOL)
            if df is None:
                time.sleep(5)
                continue

            sma_fast = df['close'].rolling(10).mean()
            sma_slow = df['close'].rolling(50).mean()
            
            curr_fast, curr_slow = sma_fast.iloc[-1], sma_slow.iloc[-1]
            prev_fast, prev_slow = sma_fast.iloc[-2], sma_slow.iloc[-2]
            price = df['close'].iloc[-1]

            print(f"₿ {datetime.now().strftime('%H:%M')} | BTC: {price:.2f} | Fast: {curr_fast:.2f} | Slow: {curr_slow:.2f}")

            # --- 3. EXECUTION ---
            if prev_fast < prev_slow and curr_fast > curr_slow:
                print("🚀 SIGNAL: BUY")
                execute_trade('buy', SYMBOL)
                time.sleep(300)
            
            elif prev_fast > prev_slow and curr_fast < curr_slow:
                print("🔻 SIGNAL: SELL")
                execute_trade('sell', SYMBOL)
                time.sleep(300)

            time.sleep(900)

    except Exception as e:
        err = f"☠️ CRASH: {e}\n{traceback.format_exc()}"
        print(err)
        send_alert(err, "error")
        mt5.shutdown()

if __name__ == "__main__":
    try:
        run_bot()
    except KeyboardInterrupt:
        mt5.shutdown()

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import time
import requests
import traceback
from datetime import datetime

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
# 1. CREDENTIALS (MT5)
LOGIN_ID = 12345678  # Replace with your ID
PASSWORD = "your_password"
SERVER = "your_server"
DEMO_ONLY = True  # Safety lock

# 2. DISCORD WEBHOOK
WEBHOOK_URL = "YOUR_DISCORD_WEBHOOK_HERE"

# 3. ASSETS TO TRADE
ASSETS = ["EURUSD", "GBPUSD", "USDJPY", "XAUUSD", "US500"]

# 4. STRATEGY SETTINGS
HTF_TIMEFRAME = mt5.TIMEFRAME_H4
LTF_TIMEFRAME = mt5.TIMEFRAME_M5
FIXED_LOT = 0.01
MAGIC_NUMBER = 777000
DEVIATION = 20
SYMBOL_COOLDOWN_SECONDS = 300 
START_HOUR = 7    
END_HOUR = 22     

# 5. COLORS
COLOR_GREEN = 5763719
COLOR_RED = 15548997
COLOR_BLUE = 3447003

# ==========================================
# 📡 MODULE 1: DISCORD & ALERTS
# ==========================================
def send_signal_card(symbol, action, entry, sl, tps, digits=2):
    if not WEBHOOK_URL or "YOUR_NEW" in WEBHOOK_URL: return
    
    # Create a dynamic format string based on digits (e.g., "{:.5f}")
    fmt = f"{{:.{digits}f}}"

    # Apply formatting to TPs, SL, and Entry
    tp_text = "".join([f"TP{i}  {fmt.format(target)}\n" for i, target in enumerate(tps, 1)])
    sl_text = fmt.format(sl)
    entry_text = fmt.format(entry)

    color = COLOR_GREEN if 'buy' in action.lower() else COLOR_RED
    
    payload = {
        "username": "Hedge Fund Bot ⚡",
        "embeds": [{
            "title": "FOREX PRO SIGNAL™",
            "description": f"#{symbol}  {action.upper()}  @{entry_text}",
            "color": color,
            "fields": [{"name": "\u200b", "value": f"```{tp_text}\nSL   {sl_text}```"}],
            "footer": {"text": f"Algo Executed • {datetime.utcnow().strftime('%H:%M UTC')}"}
        }]
    }
    try: requests.post(WEBHOOK_URL, json=payload, timeout=5)
    except: pass

def send_balance_report():
    if not WEBHOOK_URL or "YOUR_NEW" in WEBHOOK_URL: return
    account = mt5.account_info()
    if not account: return
    color = COLOR_GREEN if account.profit >= 0 else COLOR_RED
    payload = {
        "username": "Accountant 💰",
        "embeds": [{
            "title": "🏦 BALANCE CHECK",
            "color": color,
            "fields": [
                {"name": "Balance", "value": f"${account.balance:,.2f}", "inline": True},
                {"name": "Equity", "value": f"${account.equity:,.2f}", "inline": True},
                {"name": "Open PnL", "value": f"${account.profit:,.2f}", "inline": True}
            ],
            "footer": {"text": f"Account: {account.login}"}
        }]
    }
    try: requests.post(WEBHOOK_URL, json=payload, timeout=5)
    except: pass

def send_alert(msg, level="info"):
    if not WEBHOOK_URL or "YOUR_NEW" in WEBHOOK_URL: return
    colors = {"info": COLOR_BLUE, "error": COLOR_RED}
    payload = {"embeds": [{"description": msg, "color": colors.get(level), "timestamp": datetime.utcnow().isoformat()}]}
    try: requests.post(WEBHOOK_URL, json=payload, timeout=5)
    except: pass

# ==========================================
# 🔐 MODULE 2: CONNECTION
# ==========================================
def connect_mt5():
    if not mt5.initialize():
        print("❌ MT5 initialize() failed.")
        return False
    if LOGIN_ID != 0:
        if not mt5.login(LOGIN_ID, password=PASSWORD, server=SERVER):
            print(f"❌ MT5 login failed for account {LOGIN_ID}.")
            return False
    account = mt5.account_info()
    if DEMO_ONLY and account and account.trade_mode == mt5.ACCOUNT_TRADE_MODE_REAL:
        send_alert("⛔ SAFETY STOP: Real Money Account Detected!", "error")
        mt5.shutdown()
        return False
    print(f"✅ MT5 Connected: {account.login}")
    return True

# ==========================================
# 📉 MODULE 3: STRATEGY LOGIC
# ==========================================
def is_volume_healthy(symbol):
    rates = mt5.copy_rates_from_pos(symbol, LTF_TIMEFRAME, 0, 21)
    if rates is None or len(rates) < 21: return False
    df = pd.DataFrame(rates)
    avg_vol = df['tick_volume'].iloc[:-1].mean()
    if df['tick_volume'].iloc[-2] < (avg_vol * 0.5): return False
    return True

def get_fib_bias(symbol):
    rates = mt5.copy_rates_from_pos(symbol, HTF_TIMEFRAME, 0, 100)
    if rates is None or len(rates) < 100: return "neutral"
    df = pd.DataFrame(rates)
    high, low = df['high'].max(), df['low'].min()
    price = df['close'].iloc[-1]
    diff = high - low
    if price > (high - (diff * 0.618)): return "bullish"
    if price < (high - (diff * 0.5)): return "bearish"
    return "neutral"

def get_atr(symbol, timeframe, period=14):
    """Calculates the Average True Range (ATR) for dynamic stops."""
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, period + 2)
    if rates is None or len(rates) < period + 1: return 0.0
    
    df = pd.DataFrame(rates)
    df['h-l'] = df['high'] - df['low']
    df['h-pc'] = abs(df['high'] - df['close'].shift(1))
    df['l-pc'] = abs(df['low'] - df['close'].shift(1))
    df['tr'] = df[['h-l', 'h-pc', 'l-pc']].max(axis=1)
    
    return df['tr'].rolling(window=period).mean().iloc[-1]

def calculate_tps(entry, sl, action):
    risk = abs(entry - sl)
    mult = 1.0 if action == 'buy' else -1.0
    return [entry + (risk * i * mult) for i in [1.0, 2.0, 3.0]]

def count_pending_orders(symbol):
    """Checks if there are already active pending orders for this symbol."""
    orders = mt5.orders_get(symbol=symbol)
    if orders is None: return 0
    return len(orders)

# ==========================================
# ⚡ MODULE 4: EXECUTION (FIXED)
# ==========================================
def _get_filling_mode(symbol):
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None: return mt5.ORDER_FILLING_FOK
    # Check if IOC (Immediate or Cancel) is allowed (Bit 2)
    if symbol_info.filling_mode & 2:
        return mt5.ORDER_FILLING_IOC
    return mt5.ORDER_FILLING_FOK

def place_limit_order(symbol, action, limit_price, stop_loss):
    # 1. Get Symbol Details
    info = mt5.symbol_info(symbol)
    tick = mt5.symbol_info_tick(symbol)
    if not info or not tick: return

    # 2. Validate Entry Price (Prevents Error 10015)
    if action == 'buy' and limit_price >= tick.ask:
        print(f"⚠️ SKIP: Price moved. Buy Limit {limit_price} >= Ask {tick.ask}")
        return
    if action == 'sell' and limit_price <= tick.bid:
        print(f"⚠️ SKIP: Price moved. Sell Limit {limit_price} <= Bid {tick.bid}")
        return

    # 3. Fix Stops Distance (Prevents Error 10016)
    # Minimum valid distance = stops_level + 10 points buffer
    min_dist = (info.trade_stops_level + 10) * info.point 
    
    # Normalize to broker precision
    limit_price = round(limit_price, info.digits)
    stop_loss = round(stop_loss, info.digits)

    # Force Stop Loss to be valid distance
    if action == 'buy':
        if stop_loss >= limit_price or (limit_price - stop_loss) < min_dist:
             stop_loss = limit_price - min_dist
    elif action == 'sell':
        if stop_loss <= limit_price or (stop_loss - limit_price) < min_dist:
             stop_loss = limit_price + min_dist
    
    stop_loss = round(stop_loss, info.digits)

    # 4. Calculate & Validate Take Profits
    tps = calculate_tps(limit_price, stop_loss, action)
    target_tp = round(tps[1], info.digits) # Default to TP2

    if action == 'buy':
        if (target_tp - limit_price) < min_dist: target_tp = limit_price + min_dist
    else:
        if (limit_price - target_tp) < min_dist: target_tp = limit_price - min_dist
    
    target_tp = round(target_tp, info.digits)

    # 5. Send Request
    filling_mode = _get_filling_mode(symbol)
    order_type = mt5.ORDER_TYPE_BUY_LIMIT if action == 'buy' else mt5.ORDER_TYPE_SELL_LIMIT

    request = {
        "action": mt5.TRADE_ACTION_PENDING,
        "symbol": symbol,
        "volume": FIXED_LOT,
        "type": order_type,
        "price": limit_price,
        "sl": stop_loss,
        "tp": target_tp,
        "deviation": DEVIATION,
        "magic": MAGIC_NUMBER,
        "comment": "Sniper Bot",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": filling_mode, 
    }
    
    res = mt5.order_send(request)
    
    if res and res.retcode == mt5.TRADE_RETCODE_DONE:
        print(f"🎯 LIMIT PLACED: {symbol} @ {limit_price} | SL: {stop_loss}")
        # Recalculate TPs for display match
        final_tps = calculate_tps(limit_price, stop_loss, action)
        # Pass digits to fix Discord decimals
        send_signal_card(symbol, f"LIMIT {action.upper()}", limit_price, stop_loss, final_tps, digits=info.digits)
    elif res:
        print(f"❌ Limit Error: {res.comment} ({res.retcode})")

# ==========================================
# 🔄 MAIN LOOP
# ==========================================
def run_system():
    if not connect_mt5(): return
    print("✅ System Online.")
    send_balance_report()
    
    is_sleeping = False
    last_trade_times = {}
    
    try:
        while True:
            now = datetime.now()
            # Time Filter
            if not (START_HOUR <= now.hour < END_HOUR):
                if not is_sleeping:
                    send_alert("💤 Hibernating.")
                    is_sleeping = True
                time.sleep(60)
                continue
            
            if is_sleeping: is_sleeping = False
            
            for symbol in ASSETS:
                try:
                    # --- NEW SAFETY CHECK ---
                    # Prevents duplicate orders if one is already pending
                    if count_pending_orders(symbol) > 0:
                        continue
                    # ------------------------

                    # 1. Check Volume & Bias
                    if not is_volume_healthy(symbol): continue
                    
                    last_trade = last_trade_times.get(symbol)
                    if last_trade and (now - last_trade).total_seconds() < SYMBOL_COOLDOWN_SECONDS:
                        continue

                    bias = get_fib_bias(symbol)
                    if bias == "neutral": continue

                    # 2. Identify Entry Candle
                    rates = mt5.copy_rates_from_pos(symbol, LTF_TIMEFRAME, 0, 5)
                    if rates is None or len(rates) < 5: continue
                    df = pd.DataFrame(rates)

                    limit_price = 0.0
                    action = ""

                    # 3. Define Entry Logic (FVG Style)
                    if bias == "bullish":
                        if df['high'].iloc[-4] < df['low'].iloc[-2]: 
                            action = 'buy'
                            limit_price = df['high'].iloc[-4]

                    elif bias == "bearish":
                        if df['low'].iloc[-4] > df['high'].iloc[-2]:
                            action = 'sell'
                            limit_price = df['low'].iloc[-4]
                    
                    # 4. Define Stop Loss Logic (ATR Based)
                    if limit_price > 0:
                        atr = get_atr(symbol, LTF_TIMEFRAME)
                        atr_mult = 2.0  # Adjust multiplier (2.0 standard)
                        
                        sl_level = 0.0
                        if action == 'buy':
                            sl_level = limit_price - (atr * atr_mult)
                        else:
                            sl_level = limit_price + (atr * atr_mult)

                        # Execute
                        if sl_level > 0:
                            place_limit_order(symbol, action, limit_price, sl_level)
                            last_trade_times[symbol] = datetime.now()

                except Exception as e:
                    print(f"⚠️ Error {symbol}: {e}")
                    traceback.print_exc()
            
            time.sleep(60)

    except KeyboardInterrupt:
        mt5.shutdown()

if __name__ == "__main__":
    run_system()

In [2]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import requests
import time
import traceback
from datetime import datetime, timedelta

# ==========================================
# ⚙️ USER CONFIGURATION
# ==========================================

# 1. ACCOUNT CREDENTIALS (FOR AUTO-LOGIN)
# ------------------------------------------
# Leave these 0/None if you are already logged into the MT5 Terminal.
LOGIN_ID =2707383   # Example: 12345678
PASSWORD = "Ducanh@6"
SERVER = "Headway-Demo" # Example: "ICMarkets-Demo"

# 2. SECURITY & ALERTS
# ------------------------------------------
WEBHOOK_URL = "https://discord.com/api/webhooks/1463101858791952554/az4l16sYfAZ9mrlS7cQ_F8mDvkRM68Cfd4kXDss5W9LzRvoti7RdvadL32cXDIOrKOlI"
DEMO_ONLY = True         # 🔒 SAFETY LOCK: Stops bot if account is Real Money

In [ ]:

import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import requests
import time
import traceback
from datetime import datetime, timedelta

# ==========================================
# ⚙️ USER CONFIGURATION
# ==========================================

# 1. ACCOUNT CREDENTIALS (FOR AUTO-LOGIN)
# ------------------------------------------
# Leave these 0/None if you are already logged into the MT5 Terminal.
class Config:
    LOGIN_ID = 2707383   # Example: 12345678
    PASSWORD = "Ducanh@6"
    SERVER = "Headway-Demo" # Example: "ICMarkets-Demo"
    
    # 2. SECURITY & ALERTS
    # ------------------------------------------
    WEBHOOK_URL = "https://discord.com/api/webhooks/1463101858791952554/az4l16sYfAZ9mrlS7cQ_F8mDvkRM68Cfd4kXDss5W9LzRvoti7RdvadL32cXDIOrKOlI"
    DEMO_ONLY = True         # 🔒 SAFETY LOCK: Stops bot if account is Real Money

    # --- RISK & STRATEGY ---
    BASE_RISK_PCT = 0.01       # 1% Standard Risk
    GOLD_MAX_RISK_PCT = 0.05   # 5% Max Risk for Gold A+
    RISK_REWARD = 3.0          # Target 3R
    SL_BUFFER_POINTS = 50      # 5 Pip Buffer for SL

    # --- REPORTING ---
    PNL_REPORT_HOUR = 23       # Hour to send report (0-23)
    TIMEZONE_OFFSET = 7        # UTC+7 (Vietnam/Bangkok)

    # --- ASSETS ---
    ASSETS = ["XAUUSD", "EURUSD", "GBPUSD", "USDJPY", "AUDCAD", "BTCUSD"]
    TF_HTF = mt5.TIMEFRAME_H1
    TF_MED = mt5.TIMEFRAME_M15
    TF_LTF = mt5.TIMEFRAME_M5

    MAGIC_NUMBER = 999003
    DEVIATION = 20

# ==========================================
# 2. REPORTER
# ==========================================
class Reporter:
    @staticmethod
    def send_discord(content):
        if not Config.WEBHOOK_URL: return
        try: requests.post(Config.WEBHOOK_URL, json={"content": content})
        except: pass

    @staticmethod
    def send_trade_alert(symbol, action, entry, sl, tp, risk_pct, quality):
        msg = (f"🚀 **SMC ENTRY TRIGGERED**\n"
               f"**Symbol:** {symbol}\n"
               f"**Type:** {action.upper()} ({quality} Setup)\n"
               f"**Entry:** {entry} | **SL:** {sl} | **TP:** {tp}\n"
               f"**Risk:** {risk_pct*100}%")
        Reporter.send_discord(msg)

    @staticmethod
    def send_daily_summary(date_obj):
        history = mt5.history_deals_get(date_obj - timedelta(hours=24), date_obj + timedelta(hours=1))
        if not history: return
        
        daily_profit = sum([deal.profit for deal in history])
        balance = mt5.account_info().balance
        equity = mt5.account_info().equity
        
        msg = (f"📊 **DAILY PnL REPORT** ({date_obj.strftime('%Y-%m-%d')})\n"
               f"**Profit:** ${daily_profit:.2f}\n"
               f"**Balance:** ${balance:.2f}\n"
               f"**Equity:** ${equity:.2f}")
        Reporter.send_discord(msg)

# ==========================================
# 3. SMC STRATEGY LOGIC
# ==========================================
class SMCStrategy:
    @staticmethod
    def get_rsi(symbol, timeframe, period=14):
        rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, period + 15)
        if rates is None or len(rates) < period + 1: return 50.0
        df = pd.DataFrame(rates)
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi.iloc[-1]

    @staticmethod
    def detect_market_structure(symbol):
        rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 50)
        if rates is None or len(rates) < 50: return "neutral"
        df = pd.DataFrame(rates)
        highs = df['high'].rolling(5, center=True).max()
        lows = df['low'].rolling(5, center=True).min()
        
        if len(highs.dropna()) < 5: return "neutral"
        
        last_high = highs.dropna().iloc[-1]
        prev_high = highs.dropna().iloc[-5]
        last_low = lows.dropna().iloc[-1]
        prev_low = lows.dropna().iloc[-5]
        if last_high > prev_high and last_low > prev_low: return "bullish"
        elif last_high < prev_high and last_low < prev_low: return "bearish"
        return "ranging"

    @staticmethod
    def detect_fvg(symbol, timeframe, bias):
        rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 10)
        if rates is None: return None
        df = pd.DataFrame(rates)
        for i in range(len(df)-2, 2, -1):
            c1, c2, c3 = df.iloc[i-2], df.iloc[i-1], df.iloc[i]
            if bias == "bullish" and c2['close'] > c2['open']:
                if c1['high'] < c3['low']:
                    return {'type': 'bullish', 'entry': c3['low'], 'sl': c1['high']} 
            elif bias == "bearish" and c2['close'] < c2['open']:
                if c1['low'] > c3['high']:
                    return {'type': 'bearish', 'entry': c3['high'], 'sl': c1['low']} 
        return None

    @staticmethod
    def detect_amd_phase(symbol):
        rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M15, 0, 20)
        if rates is None: return "unknown"
        df = pd.DataFrame(rates)
        high_low_diff = df['high'] - df['low']
        avg_range = high_low_diff.mean()
        recent_vol = high_low_diff.iloc[-10:].mean()
        
        if recent_vol < (avg_range * 0.7): return "Accumulation"
        
        range_low = df['low'].iloc[-20:-5].min()
        last_low = df['low'].iloc[-1]
        last_close = df['close'].iloc[-1]
        if last_low < range_low and last_close > range_low: return "Manipulation"
        
        range_high = df['high'].iloc[-20:-5].max()
        last_high = df['high'].iloc[-1]
        if last_high > range_high and last_close < range_high: return "Manipulation"
        
        return "Distribution"

# ==========================================
# 4. EXECUTOR (TICK SIZE FIX)
# ==========================================
class Executor:
    @staticmethod
    def connect():
        if not mt5.initialize():
            print("❌ MT5 Init Failed")
            return False
        if Config.LOGIN_ID != 12345678:
            return mt5.login(Config.LOGIN_ID, Config.PASSWORD, Config.SERVER)
        return True

    @staticmethod
    def normalize_price(symbol, price):
        """
        Rounds price to the nearest tick size (e.g., 0.25 for US500).
        Fixes Error 10015.
        """
        symbol_info = mt5.symbol_info(symbol)
        if symbol_info is None: return price
        
        tick_size = symbol_info.trade_tick_size
        if tick_size > 0:
            # Round to nearest tick step
            price = round(price / tick_size) * tick_size
        
        # Cleanup floating point artifacts (e.g. 4000.2500001 -> 4000.25)
        return round(price, symbol_info.digits)

    @staticmethod
    def calculate_lot_size(symbol, entry, sl, risk_percent):
        info = mt5.symbol_info(symbol)
        account = mt5.account_info()
        if not info or not account: return 0.0
        
        balance = account.balance
        risk_usd = balance * risk_percent
        dist = abs(entry - sl)
        if dist == 0: return 0.0
        
        tick_value = info.trade_tick_value
        if tick_value == 0: tick_value = 1.0 
        
        raw_lot = risk_usd / ((dist / tick_size) * tick_value)
        step = info.volume_step
        lot = round(raw_lot / step) * step
        lot = max(info.volume_min, min(info.volume_max, lot))
        
        if lot < info.volume_min: return 0.0
        return lot

    @staticmethod
    def execute_trade(symbol, signal_type, entry, sl, quality="B"):
        # 1. Risk Calc
        risk_pct = Config.BASE_RISK_PCT
        if "XAU" in symbol and quality == "A+": risk_pct = Config.GOLD_MAX_RISK_PCT
        elif "XAU" in symbol: risk_pct = 0.02

        # 2. Buffer & Normalize
        info = mt5.symbol_info(symbol)
        buffer = Config.SL_BUFFER_POINTS * info.point
        final_sl = sl - buffer if signal_type == 'buy' else sl + buffer
        
        # --- CRITICAL FIX: Tick Size Normalization ---
        entry = Executor.normalize_price(symbol, entry)
        final_sl = Executor.normalize_price(symbol, final_sl)

        # 3. Targets (TP)
        dist = abs(entry - final_sl)
        tp = entry + (dist * Config.RISK_REWARD) if signal_type == 'buy' else entry - (dist * Config.RISK_REWARD)
        tp = Executor.normalize_price(symbol, tp)
        
        # 4. Sizing
        lot_size = Executor.calculate_lot_size(symbol, entry, final_sl, risk_pct)
        if lot_size == 0.0: 
            print(f"⚠️ {symbol}: Balance too low or Lot 0"); return

        # 5. Order
        req = {
            "action": mt5.TRADE_ACTION_PENDING,
            "symbol": symbol,
            "volume": lot_size,
            "type": mt5.ORDER_TYPE_BUY_LIMIT if signal_type == 'buy' else mt5.ORDER_TYPE_SELL_LIMIT,
            "price": entry,
            "sl": final_sl,
            "tp": tp,
            "deviation": Config.DEVIATION,
            "magic": Config.MAGIC_NUMBER,
            "comment": f"SMC-{quality}",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_RETURN,
        }
        res = mt5.order_send(req)
        
        if res.retcode == mt5.TRADE_RETCODE_DONE:
            print(f"🚀 {symbol} {signal_type.upper()} | Lot: {lot_size} | Entry: {entry}")
            Reporter.send_trade_alert(symbol, signal_type, entry, final_sl, tp, risk_pct, quality)
        else:
            print(f"❌ Error {symbol}: {res.comment} (Retcode: {res.retcode})")

# ==========================================
# 5. MAIN ENGINE
# ==========================================
def run():
    if not Executor.connect(): 
        print("❌ Connect Failed - Check Login ID")
        return
        
    print(f"✅ SMC Bot Online | PnL Report at {Config.PNL_REPORT_HOUR}:00 UTC+7")
    
    last_summary_date = None

    try:
        while True:
            # --- NON-TRADING ---
            utc_now = datetime.utcnow()
            local_now = utc_now + timedelta(hours=Config.TIMEZONE_OFFSET)
            
            if local_now.hour == Config.PNL_REPORT_HOUR and last_summary_date != local_now.date():
                print("📊 Sending Daily PnL...")
                Reporter.send_daily_summary(local_now)
                last_summary_date = local_now.date()

            # --- TRADING ---
            for symbol in Config.ASSETS:
                try:
                    # 1. Structure
                    structure = SMCStrategy.detect_market_structure(symbol)
                    phase = SMCStrategy.detect_amd_phase(symbol)
                    rsi_h1 = SMCStrategy.get_rsi(symbol, mt5.TIMEFRAME_H1)
                    rsi_m5 = SMCStrategy.get_rsi(symbol, mt5.TIMEFRAME_M5)
                    
                    signal, quality = None, "B"
                    
                    # 2. Bullish
                    if structure == "bullish" and rsi_h1 > 50 and rsi_m5 < 45:
                        fvg = SMCStrategy.detect_fvg(symbol, mt5.TIMEFRAME_M15, "bullish")
                        if fvg:
                            if "Manipulation" in phase: quality = "A+"
                            Executor.execute_trade(symbol, 'buy', fvg['entry'], fvg['sl'], quality)

                    # 3. Bearish
                    elif structure == "bearish" and rsi_h1 < 50 and rsi_m5 > 55:
                        fvg = SMCStrategy.detect_fvg(symbol, mt5.TIMEFRAME_M15, "bearish")
                        if fvg:
                            if "Manipulation" in phase: quality = "A+"
                            Executor.execute_trade(symbol, 'sell', fvg['entry'], fvg['sl'], quality)
                            
                except Exception as e:
                    print(f"Err {symbol}: {e}")
            
            time.sleep(60)

    except KeyboardInterrupt:
        mt5.shutdown()
        print("🛑 Bot Stopped")

if __name__ == "__main__":
    run()

✅ SMC Bot Online | PnL Report at 23:00 UTC+7
📊 Sending Daily PnL...
